# ORBIT — Gate 7: Real-Experience Policy Learning & Value Calibration

**Milestone:** Gate 7  
**Controlling Specification:** `ORBIT_IMPLEMENTATION_BRIEF_v1.3.md` (Gate 7: Sections 394–401)  
**Objective:** Build and evaluate an offline, behavior-regularized policy baseline (AWR with entropy regularization) using **real transitions only** (zero imagined rollouts).

### Core Pipeline:
1. **Dataset Audit:** Validates real experience replay (`augmented_replay_84eps.npz`) for temporal alignment ($T_{\text{obs}} = T_{\text{act}} + 1$), bounds, and strict seed isolation.
2. **Freeze World Model:** Strictly freeze DINOv2 visual encoder backbone, spatial adapter, predictor, belief core, and transition heads.
3. **Pre-Training Calibration:** Validates reward prediction and value estimates against real transitions and returns on held-out seeds (`3030..3049`).
4. **Offline AWR Policy Learning:** 300 updates on real transitions using advantage-weighted regression with entropy regularization.
5. **Post-Training Calibration:** Re-evaluates reward prediction and value calibration.
6. **Rigorous Crafter Benchmark:** 30 pinned seeds (`5000..5029`) comparing Random Baseline vs Gate 6 Champion ($H=4$) vs Gate 7 Candidate.
7. **Promotion Decision Matrix:** Formal evaluation of baseline establishment vs champion promotion.

### Cell 1: Environment & Hardware Diagnostics
Verifies CUDA availability, GPU architecture, VRAM, and PyTorch configuration.

In [ ]:
# Cell 1: Hardware Diagnostics
import os
import sys
import platform
import torch

print("=" * 75)
print("  SYSTEM & HARDWARE DIAGNOSTICS")
print("=" * 75)
print(f"Python Version:      {platform.python_version()}")
print(f"PyTorch Version:     {torch.__version__}")
print(f"CUDA Available:      {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:         {torch.cuda.get_device_name(0)}")
    print(f"Device Count:        {torch.cuda.device_count()}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Total VRAM:          {mem:.2f} GB")
print("=" * 75)

### Cell 2: Google Drive Mounting & Namespace Setup
Mounts Google Drive and ensures directory structure exists under `/content/drive/MyDrive/learn-a-world/orbit/`.

In [ ]:
# Cell 2: Google Drive Mounting
import os
from pathlib import Path

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
        print("[Drive Mount] Google Drive successfully mounted.")
    else:
        print("[Drive Mount] Google Drive already mounted.")
except ImportError:
    print("[Local Mode] Google Colab drive module not found; running locally.")

drive_candidates = [
    Path('/content/drive/MyDrive/learn-a-world/orbit'),
    Path('/content/drive/My Drive/learn-a-world/orbit'),
]
drive_orbit_dir = None
for d in drive_candidates:
    if d.parent.exists():
        drive_orbit_dir = d
        d.mkdir(parents=True, exist_ok=True)
        (d / 'checkpoints').mkdir(parents=True, exist_ok=True)
        (d / 'data').mkdir(parents=True, exist_ok=True)
        (d / 'results' / 'gate7').mkdir(parents=True, exist_ok=True)
        print(f"[Persistent Storage] Configured Google Drive tree at: {drive_orbit_dir}")
        break

### Cell 3: Clean Repository Sync & Dependency Setup
Fetches latest upstream `main` and installs Crafter.

In [ ]:
# Cell 3: Repository Sync
%cd /content
import os
from pathlib import Path

repo_path = Path('/content/learn-a-world')
if (repo_path / '.git').exists():
    %cd /content/learn-a-world
    !git fetch origin main
    !git reset --hard origin/main
else:
    !git clone https://github.com/sahiltolani30/learn-a-world.git /content/learn-a-world
    %cd /content/learn-a-world

!pip install -q crafter matplotlib
print("[Repo Sync] Repository up-to-date on branch main.")

### Cell 4: Checkpoint & Dataset Verification
Verifies SHA-256 hashes of the World Model Champion (`world_model_phase3.pt`) and Production Policy Champion (`policy_h4.pt.gz`).

In [ ]:
# Cell 4: Checkpoint Verification
import hashlib
from pathlib import Path

checkpoints = {
    "World Model Champion": Path("results/phase3/world_model_phase3.pt"),
    "Gate 6 Policy Champion": Path("results/gate6/policy_h4.pt.gz"),
}

for name, p in checkpoints.items():
    if p.exists():
        sha = hashlib.sha256(p.read_bytes()).hexdigest()
        print(f"✓ {name:25s}: {p.stat().st_size / (1024*1024):.1f} MB | SHA-256: {sha}")
    else:
        print(f"✗ {name:25s}: NOT FOUND at {p}")

### Cell 5: Execute Gate 7 — Real-Experience Policy Training & 30-Seed Benchmark
Executes the complete Gate 7 runner: Replay Audit -> Pre-Train Calibration -> 300 Offline AWR Updates -> Post-Train Calibration -> 30-Seed Crafter Benchmark -> Drive Sync.

In [ ]:
# Cell 5: Run Gate 7 Policy Pipeline
%cd /content/learn-a-world

!PYTHONPATH=. python3 scripts/run_gate7_real_policy.py \
    --world-model results/phase3/world_model_phase3.pt \
    --replay-dataset data/phase3/augmented_replay_84eps.npz \
    --heldout-eval data/gate3/eval_heldout_seeds.pt.gz \
    --gate6-champion results/gate6/policy_h4.pt.gz \
    --updates 300 \
    --batch-size 8 \
    --seq-len 8 \
    --actor-lr 1e-4 \
    --value-lr 1e-4 \
    --gamma 0.99 \
    --advantage-temperature 1.0 \
    --max-weight 20.0 \
    --entropy-weight 0.01 \
    --benchmark-seeds 30 \
    --max-steps 300 \
    --device auto \
    --output-dir results/gate7

### Cell 6: Visual Acceptance Report & Metrics Summary
Displays the 4-panel Gate 7 telemetry plot and prints the official decision matrix.

In [ ]:
# Cell 6: Visual Report & Metrics
import json
from IPython.display import Image, display
from pathlib import Path

metrics_path = Path("results/gate7/gate7_experiment_metrics.json")
plots_path = Path("results/gate7/gate7_telemetry_plots.png")

if plots_path.exists():
    display(Image(filename=str(plots_path)))

if metrics_path.exists():
    with open(metrics_path) as f:
        m = json.load(f)
    print("=" * 80)
    print(f"GATE 7 FINAL DECISION: {m.get('promotion_status')} (Baseline: {m.get('baseline_status')})")
    print("=" * 80)
    b = m.get('benchmark_results', {})
    for k, v in b.items():
        print(f"{k:20s} | Return: {v.get('mean_return', 0.0):+.2f} ± {v.get('std_return', 0.0):.2f} | Score: {v.get('crafter_score', 0.0):.2f}% | Steps: {v.get('mean_length', 0.0):.1f}")
    print("=" * 80)